In [1]:
x1, y1 = 1,8
x2, y2 = 2,11
x3, y3 = 3,16

In [47]:
# For single feature linear regression.
def predict(x, m, b):
    return m * x + b

In [49]:
# For single feature linear regression, we can calculate the total error as follows:
def total_err_sqr(m,c):
    e1 = y1 - predict(x1, m, c)
    e2 = y2 - predict(x2, m, c)
    e3 = y3 - predict(x3, m, c)

    return ((e1*e1 + e2*e2 + e3*e3)/3)**0.5

## Bruteforce

Fixing m and checking for all c, then increase m repeate logic. where m,c is in range of [l,r]. We are moving m and c in same direction.

In [ ]:
def find_mc1(left,right):
    best = (0,0)
    lowest_err = None
    m1, m2, c = left,left,left
    
    while m1<=right:
        m2 = left
        c = left
        while c<=right:
            ttl_err = total_err_sqr(m1, m2, c)
            if lowest_err is None or ttl_err < lowest_err:
                best = (m1, m2, c)
                lowest_err = ttl_err
            c +=0.1
        m1 +=0.1

    return print(f"best (m1, m2, c) => {best}\nlowest error is: {lowest_err}")

In [6]:
find_mc1(-100,100)

best (m,c) => (3.999999999998598, 3.6999999999985977)
lowest error is: 0.4725815626249637


<hr>

## Moving with rate of change

Moving m and c to the direction where we get lowest change in error. Here value of initial m and c is (0,0) and we are moving m and c 'stp' steps. We are moving m and c in same direction.

In [7]:
def gradients_1direction(m, c, h=0.00001):
    e = total_err_sqr(m, c)
    e_after_moving = total_err_sqr(m + h, c + h)
    d = (e_after_moving - e) / h

    return d

In [8]:
def find_mc2(m, c, step, iterations):
    best = (m,c)
    lowest_err = float('inf')
    
    for i in range(iterations):
        e = total_err_sqr(m, c)
        delta = gradients_1direction(m, c)
        # print(f"{i}: m={m:.5f}, c={c:.5f}, delta={delta:.5f}, error={e:.5f}")

        if e < lowest_err:
            best = (m, c)
            lowest_err = e

        if delta < 0:
            m, c = m+step, c+step
        
        elif delta > 0:
            m, c = m-step, c-step

    print(f"best (m,c) => {best}\nlowest error is: {lowest_err}")
    return m, c

In [9]:
find_mc2(0,0,0.1,500)

best (m,c) => (3.900000000000002, 3.900000000000002)
lowest error is: 0.4795831523312721


(3.800000000000002, 3.800000000000002)

Moving m and c to the direction where we get lowest change in error. Here value of initial m and c is (0, 0) and we are moving m and c 'stp' steps. We are moving m and c independently in different direction.

In [51]:
# For single feature linear regression, we can calculate the gradients as follows:
def gradients(m, c, h=0.00001):
    e = total_err_sqr(m, c)
    dm = (total_err_sqr(m + h, c) - e) / h
    dc = (total_err_sqr(m, c + h) - e) / h

    return dm, dc

In [24]:
def find_mc3(c, m, step, iterations):
    best = (m, c)
    lowest_err = float('inf')
    i = 0
    
    for i in range(iterations):
        delta_m, delta_c = gradients(m, c)
        e = total_err_sqr(m, c)
        
        if e < lowest_err:
            best = (m, c)
            lowest_err = e

        if delta_m < 0:
            m = m + step
        elif delta_m > 0:
            m = m - step
        
        if delta_c < 0:
            c = c + step
        elif delta_c > 0:
            c = c - step

    print(f"best (m,c) => {best}\nlowest error is: {lowest_err}")
    return m, c

In [25]:
find_mc3(0,0,0.1,500)

best (m,c) => (3.900000000000002, 3.900000000000002)
lowest error is: 0.4795831523312721


(3.800000000000002, 3.800000000000002)

In [53]:
# For single feature linear regression, we can fit the model using gradient descent as follows:
def fit(m, c, learning_rate, epochs):
    for epoch in range(epochs):
        dm, dc = gradients(m, c)

        m = m - learning_rate * dm
        c = c - learning_rate * dc

    print(f"best (m,c) => {m, c}\nlowest error is: {total_err_sqr(m,c)}"),
    return (m, c)

In [41]:
m, c = fit(m=0,c=0,learning_rate=0.1, epochs=500)

best (m,c) => (3.9999839635240786, 3.666692656656423)
lowest error is: 0.4714045210121243


In [42]:
X_test = [4, 10]
print(X_test[0])
p1 = predict(X_test[0], m, c)
print(p1)
p2 = predict(X_test[1], m, c)
print(p2)

4
19.666628510752737
43.66653229189721


In [43]:
def total_err_sqr(m,c):
    e1 = 20 - predict(X_test[0], m, c)
    e2 = 42 - predict(X_test[1], m, c)

    return ((e1*e1 + e2*e2 )/2)**0.5

In [44]:
error_test = total_err_sqr(m, c)

In [45]:
error_test

1.2017625451350848

## Let's scale this for n number of features and samples

In [1]:
# For multiple features linear regression, we can use the following function to make predictions.
def predict(x, m, c):
    y_pred = c

    for feature, weight in zip(x, m):
        y_pred += feature * weight

    return y_pred

# y_pred = c + x1*m1 + x2*m2 + x3*m3 ... xn*mn

In [2]:
# For multiple features linear regression, we can calculate the total error as follows:
def total_err_sqr(X, y, m, c):
    total = 0

    for x_row, actual in zip(X, y):
        pred = predict(x_row, m, c)
        error = actual - pred
        total += error * error

    return (total / len(y)) ** 0.5

In [3]:
# For multiple features linear regression, we can calculate the gradients as follows:
def gradients(X, y, m, c, h=0.00001):
    current_error = total_err_sqr(X, y, m, c)
    weight_gradients = []

    for i in range(len(m)):
        temp_m = m.copy()
        temp_m[i] += h

        grad = (total_err_sqr(X, y, temp_m, c) - current_error) / h
        weight_gradients.append(grad)

    c_grad = (total_err_sqr(X, y, m, c + h) - current_error) / h

    return weight_gradients, c_grad

In [4]:
# For multiple features linear regression, we can fit the model using gradient descent as follows:
def fit(X, y, learning_rate=0.001, epochs=1000):
    n_features = len(X[0])
    m = [0] * n_features
    c = 0

    for epoch in range(epochs):
        dw, db = gradients(X, y, m, c)
        for i in range(len(m)):
            m[i] -= learning_rate * dw[i]
        c -= learning_rate * db

    print("m:", m)
    print("c:", c)
    print("Error:", total_err_sqr(X, y, m, c))

    return m, c

In [5]:
X = [
    [1, 2],
    [2, 1],
    [3, 4],
]

y = [8, 11, 16, 19]

m, c = fit(X,y,learning_rate=0.01,epochs=500)

m: [3.3371667810420114, 1.1392319173927357]
c: 2.068704983692382
Error: 0.6623635305334029


In [6]:
p1 = predict([4, 3], m, c)
print(p1)
p2 = predict([10, 5], m, c)
print(p2)

18.835067860038635
41.13653238107618


In [7]:
X_test = [
    [4, 3],
    [10, 5]
]

total_err_sqr(X_test, [18.89, 40.78], m, c)

0.2550812407416354

We can see the pridect function:
```
def predict(x, m, c):
    y_pred = c

    for feature, weight in zip(x, m):
        y_pred += feature * weight

    return y_pred
```

is eventually doing = **y_pred = x0*m0 + x1*m1 + x2*m2 ... xn*mn + c**

If we want to make the model complex we have to make this prdiction function complex, let's say we want to make this fuction to be:

**y_pred = x0*m0 + x1*m1 + x0*x1*m2 + x2*m3 + x3*m4 + x2*x3*m5 ... +c**

or

**y_pred = x0*m0 + x1*m1 + log(x1*x2)*m2 + x2*m3 + x3*m4 + log(x2*x3)*m5 ... +c**

We can just add a new feature to our X_train:

for **y_pred = x0*m0 + x1*m1 + x0*x1*m2 + x2*m3 + x3*m4 + x2*x3*m5 ... +c**

> we can add x0*x1, x2*x3 and so on as new features

for **y_pred = x0*m0 + x1*m1 + log(x1*x2)*m2 + x2*m3 + x3*m4 + log(x2*x3)*m5 ... +c**

> we can add log(x0*x1), log(x2*x3) and so on as new features

In [67]:
X = [
    [1, 2],
    [2, 1],
    [3, 4],
]

y = [8, 11, 16, 19]

In [8]:
def add_feature(X, feature_fn):
    X_new = []
    for row in X:
        new_feature = feature_fn(row)

        X_new.append(
            row + [new_feature]
        )

    return X_new

In [9]:
X_interaction = add_feature(
    X,
    lambda row: row[0] * row[1]
)

for row in X_interaction:
    print(row)

[1, 2, 2]
[2, 1, 2]
[3, 4, 12]


In [10]:
m, c = fit(X_interaction,y,learning_rate=0.01,epochs=500)

m: [3.5859893406899324, 1.448895377045123, -0.2374021176414498]
c: 2.331674009384943
Error: 0.3122973988646108


In [11]:
import math

X_log_interaction = add_feature(
    X,
    lambda row: math.log(row[0] * row[1])
)

for row in X_log_interaction:
    print(row)

[1, 2, 0.6931471805599453]
[2, 1, 0.6931471805599453]
[3, 4, 2.4849066497880004]


In [12]:
m, c = fit(X_log_interaction,y,learning_rate=0.01,epochs=500)

m: [3.1425616409362918, 0.9942351724938625, 0.4817112530502987]
c: 2.107592062761454
Error: 0.7619006440199511
